# ============================================================
# DigiCow adoption challenge pipeline
# ============================================================

In [ ]:
system("pip -q install -U gdown")
system("gdown --fuzzy 'https://drive.google.com/file/d/1BLOgcK5YyEQXVvIwAwGZC72z_hso86qb/view?usp=sharing' -O packages.zip")
system("gdown --fuzzy 'https://drive.google.com/file/d/1Zc8RNhsx1hnkLb2sqQfgsPDyTBYsK2h7/view?usp=sharing' -O data.zip")
system('unzip "/content/packages.zip" -d "/usr/local/lib/R"')
system('unzip "/content/data.zip" -d "/content"')

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(lubridate)
  library(Matrix)
  library(text2vec)
  library(irlba)
  library(lightgbm)
  library(xgboost)
  library(jsonlite)
})

# -----------------------------
# Helpers
# -----------------------------

In [ ]:
lex_encode0 <- function(v) {
  x <- as.character(v)
  x[is.na(x)] <- "nan"
  lv <- sort(unique(x))
  as.integer(factor(x, levels = lv)) - 1L
}

flatten_any <- function(z) {
  if (is.null(z)) return(character(0))
  if (!is.list(z)) return(as.character(z))
  as.character(unlist(z, recursive = TRUE, use.names = FALSE))
}

unpack_topics <- function(x) {
  if (is.na(x)) return(character(0))
  if (!is.character(x)) return(as.character(x))
  if (!startsWith(x, "[")) return(as.character(x))

  parsed <- tryCatch({
    s <- x
    s <- gsub("\\\\'", "'", s, perl = TRUE)
    s <- gsub("'", "\"", s, fixed = TRUE)
    s <- gsub("None", "null", s, fixed = TRUE)
    jsonlite::fromJSON(s, simplifyVector = FALSE)
  }, error = function(e) NULL)

  if (is.null(parsed)) return(character(0))
  flatten_any(parsed)
}

attach_trainer_history <- function(dt_all, dt_prior, y_name) {
  base_mean <- mean(dt_prior[[y_name]], na.rm = TRUE)
  C <- 15

  stat_dt <- dt_prior[
    ,
    .(pos = sum(get(y_name), na.rm = TRUE), cnt = .N),
    by = .(trainer, training_day)
  ]
  setorder(stat_dt, trainer, training_day)
  stat_dt[, pos_cum := cumsum(pos), by = trainer]
  stat_dt[, cnt_cum := cumsum(cnt), by = trainer]
  stat_dt[, pos_cum := shift(pos_cum, 1L, fill = 0), by = trainer]
  stat_dt[, cnt_cum := shift(cnt_cum, 1L, fill = 0), by = trainer]
  stat_dt[, trn_hist_sm07 := (pos_cum + C * base_mean) / (cnt_cum + C)]
  stat_dt <- stat_dt[, .(trainer, training_day, trn_hist_sm07)]

  setkey(stat_dt, trainer, training_day)
  setkey(dt_all, trainer, training_day)
  out <- stat_dt[dt_all, on = .(trainer, training_day), roll = Inf]
  setDT(out)
  out
}

# -----------------------------
# Vectorized tokenizers for text2vec
# -----------------------------

# Word tokenizer with grams, vectorized: input is character vector, output is list
tok_word_12_vec <- function(x) {
  x <- as.character(x)
  x[is.na(x)] <- ""
  lapply(x, function(s) {
    s <- tolower(s)
    m <- gregexpr("\\b\\w\\w+\\b", s, perl = TRUE)
    uni <- regmatches(s, m)[[1]]
    if (length(uni) < 2) return(uni)
    bi <- paste(uni[-length(uni)], uni[-1], sep = " ")
    c(uni, bi)
  })
}

# Char ngrams vectorized: input is character vector, output is list
tok_char_35_vec <- function(x) {
  x <- as.character(x)
  x[is.na(x)] <- ""
  lapply(x, function(s) {
    s <- tolower(s)
    n <- nchar(s, type = "chars", allowNA = FALSE)
    if (n < 3) return(character(0))
    out <- character(0)
    for (k in 3:5) {
      if (n >= k) {
        starts <- 1:(n - k + 1)
        out <- c(out, substring(s, starts, starts + k - 1))
      }
    }
    out
  })
}

# -----------------------------
# TF-IDF with full-row restoration
# -----------------------------
expand_dtm_rows <- function(dtm, doc_ids_full) {
  if (is.null(rownames(dtm))) {
    # If no rownames, assume already aligned
    return(dtm)
  }
  keep_ids <- rownames(dtm)
  row_map <- match(keep_ids, doc_ids_full)
  if (anyNA(row_map)) stop("DTM id mismatch while expanding rows.")

  n_full <- length(doc_ids_full)
  if (nrow(dtm) == n_full) {
    dtm2 <- dtm
    dimnames(dtm2) <- list(NULL, NULL)
    return(dtm2)
  }

  trip <- summary(dtm) # i,j,x with 1-based row/col indices for current dtm
  new_i <- row_map[trip$i]

  out <- sparseMatrix(
    i = new_i,
    j = trip$j,
    x = trip$x,
    dims = c(n_full, ncol(dtm)),
    dimnames = list(NULL, NULL)
  )
  as(out, "dgCMatrix")
}

make_tfidf <- function(text_vec, tokenizer_fun_vec, max_terms, min_df) {
  text_vec <- as.character(text_vec)
  text_vec[is.na(text_vec)] <- ""
  doc_ids <- as.character(seq_along(text_vec))

  it <- itoken(
    text_vec,
    tokenizer = tokenizer_fun_vec,
    progressbar = FALSE,
    ids = doc_ids
  )

  v <- create_vocabulary(it)
  v <- v[v$doc_count >= min_df, , drop = FALSE]
  setDT(v)

  # max_features by term_count desc, tie-break term asc
  setorder(v, -term_count, term)
  if (nrow(v) > max_terms) v <- v[1:max_terms]
  # final order lexicographic
  setorder(v, term)

  vec <- vocab_vectorizer(v)
  dtm <- create_dtm(it, vec)
  dtm <- as(dtm, "dgCMatrix")

  # Restore dropped empty docs to full N
  dtm <- expand_dtm_rows(dtm, doc_ids)

  # Strip attributes that can break Matrix ops
  dtm@x <- unname(as.numeric(dtm@x))
  dtm@i <- unname(dtm@i)
  dtm@p <- unname(dtm@p)
  dimnames(dtm) <- list(NULL, NULL)

  n_docs <- nrow(dtm)
  df <- as.numeric(Matrix::colSums(dtm != 0))
  idf <- log((1 + n_docs) / (1 + df)) + 1

  tfidf <- dtm %*% Diagonal(x = idf)
  tfidf <- as(tfidf, "dgCMatrix")
  tfidf@x <- unname(as.numeric(tfidf@x))
  dimnames(tfidf) <- list(NULL, NULL)

  rs <- sqrt(Matrix::rowSums(tfidf ^ 2))
  rs[rs == 0] <- 1
  tfidf <- Diagonal(x = 1 / rs) %*% tfidf
  tfidf <- as(tfidf, "dgCMatrix")
  tfidf@x <- unname(as.numeric(tfidf@x))
  dimnames(tfidf) <- list(NULL, NULL)

  tfidf
}

flip_svd_sign <- function(u, v) {
  for (j in seq_len(ncol(v))) {
    idx <- which.max(abs(v[, j]))
    if (v[idx, j] < 0) {
      v[, j] <- -v[, j]
      u[, j] <- -u[, j]
    }
  }
  list(u = u, v = v)
}

count_recent <- function(dates, window_days) {
  n <- length(dates)
  out <- integer(n)
  if (n <= 1) return(out)
  left <- 1L
  for (i in 2:n) {
    while (left < i && as.integer(dates[i] - dates[left]) >= window_days) left <- left + 1L
    out[i] <- i - left
  }
  out
}

prob_sharpen <- function(df, gamma = 1.2, eps = 1e-6) {
  cols <- setdiff(names(df), "ID")
  for (cc in cols) {
    p <- pmin(pmax(as.numeric(df[[cc]]), eps), 1 - eps)
    df[[cc]] <- (p^gamma) / ((p^gamma) + ((1 - p)^gamma))
  }
  df
}

make_group_folds <- function(groups, k = 4L, seed = 1L) {
  ug <- unique(groups)
  set.seed(seed)
  ug <- sample(ug, length(ug), replace = FALSE)
  fold_id <- rep_len(seq_len(k), length(ug))
  split(ug, fold_id)
}

sanitize_nm <- function(s) {
  x <- gsub(" ", "_", s, fixed = TRUE)
  x <- gsub(",", "", x, fixed = TRUE)
  x <- gsub("-", "_", x, fixed = TRUE)
  substr(x, 1L, 30L)
}

# -----------------------------
# 1) Load data
# -----------------------------

In [ ]:

base_dir <- "/content"

dt_tr <- fread(file.path(base_dir, "Train.csv"))
dt_te <- fread(file.path(base_dir, "Test.csv"))
dt_pr <- fread(file.path(base_dir, "Prior.csv"))

dt_tr[, training_day := as.IDate(training_day)]
dt_te[, training_day := as.IDate(training_day)]
dt_pr[, training_day := as.IDate(training_day)]

targets <- c("adopted_within_07_days", "adopted_within_90_days", "adopted_within_120_days")
cat_cols <- c(
  "gender", "age", "registration", "belong_to_cooperative",
  "county", "subcounty", "ward", "trainer", "group_name", "has_topic_trained_on"
)

dt_tr[, is_train := 1L]
dt_te[, is_train := 0L]

all_dt <- rbindlist(list(dt_tr, dt_te), fill = TRUE)
all_dt[, tp_vec := lapply(topics_list, unpack_topics)]

dt_pr[, tp_vec := lapply(topics_list, unpack_topics)]
dt_pr[, is_train := 1L]

all_dt <- rbindlist(list(all_dt, dt_pr), fill = TRUE)
setDT(all_dt)
all_dt <- copy(all_dt)
all_dt[, rid___ := .I]

all_dt <- attach_trainer_history(all_dt, dt_pr, "adopted_within_07_days")
setDT(all_dt)
all_dt <- copy(all_dt)
setorder(all_dt, training_day)

# -----------------------------
# 2) Topic-driven features
# -----------------------------

In [ ]:
all_topics <- unlist(all_dt$tp_vec, use.names = FALSE)
all_topics <- all_topics[!is.na(all_topics)]
tp_freq <- sort(table(all_topics), decreasing = TRUE)

test_topics <- unique(unlist(all_dt[is_train == 0L]$tp_vec, use.names = FALSE))
test_topics <- test_topics[!is.na(test_topics)]

top50 <- names(tp_freq)[seq_len(min(50L, length(tp_freq)))]
top50_in_test <- top50[top50 %in% test_topics]

for (tp in top50_in_test) {
  nm <- paste0("tp_f_", sanitize_nm(tp))
  set(all_dt, j = nm, value = vapply(all_dt$tp_vec, function(v) as.integer(tp %in% v), integer(1)))
}

all_dt[, tp_div_ratio := vapply(tp_vec, function(v) {
  if (length(v) == 0) return(0)
  length(unique(v)) / length(v)
}, numeric(1))]

freq_lookup <- as.integer(tp_freq)
names(freq_lookup) <- names(tp_freq)

all_dt[, tp_pop_mean := vapply(tp_vec, function(v) {
  if (length(v) == 0) return(0)
  mean(freq_lookup[v], na.rm = TRUE)
}, numeric(1))]

all_dt[, prev_id_tmp := shift(ID, 1L), by = farmer_name]
all_dt[, farm_prev_seen := cumsum(!is.na(prev_id_tmp))]
all_dt[, prev_id_tmp := NULL]

all_dt[, tp_text_blob := vapply(tp_vec, function(v) paste(v, collapse = ", "), character(1))]
all_dt[is.na(tp_text_blob), tp_text_blob := ""]

# -----------------------------
# 3) TF-IDF derived features (stable)
# -----------------------------

In [ ]:
rid_vec <- as.integer(all_dt[["rid___"]])
txt_vec <- as.character(all_dt[["tp_text_blob"]])
txt_vec[is.na(txt_vec)] <- ""

# Word TF-IDF: max_features=20, (1,2)-grams, min_df=5
tf_w <- make_tfidf(txt_vec, tok_word_12_vec, max_terms = 20L, min_df = 5L)
tf_w_mat <- as.matrix(tf_w)

w_feat <- data.table(rid___ = rid_vec)
for (j in seq_len(ncol(tf_w_mat))) {
  if (j == 8L) next  # skip python index 7
  set(w_feat, j = sprintf("w_tfx_%02d", j - 1L), value = tf_w_mat[, j])
}
all_dt <- merge(all_dt, w_feat, by = "rid___", all.x = TRUE, sort = FALSE)

# Char TF-IDF: max_features=8, char 3-5, min_df=5
tf_c <- make_tfidf(txt_vec, tok_char_35_vec, max_terms = 8L, min_df = 5L)
tf_c_mat <- as.matrix(tf_c)

c_feat <- data.table(rid___ = rid_vec)
keep_c <- c(1L, 2L, 3L, 7L, 8L)  # python [0,1,2,6,7]
keep_c <- keep_c[keep_c <= ncol(tf_c_mat)]
for (j in keep_c) {
  set(c_feat, j = sprintf("c_tfx_%02d", j - 1L), value = tf_c_mat[, j])
}
all_dt <- merge(all_dt, c_feat, by = "rid___", all.x = TRUE, sort = FALSE)

# Big TF-IDF for SVD themes: max_features=100
tf_big <- make_tfidf(txt_vec, tok_word_12_vec, max_terms = 100L, min_df = 1L)

set.seed(42)
sv <- irlba(tf_big, nv = 5L, nu = 5L)
sv2 <- flip_svd_sign(sv$u, sv$v)
sv_feats <- sweep(sv2$u, 2, sv$d, `*`)  # U * Sigma like sklearn

sv_feat <- data.table(rid___ = rid_vec)
for (k in seq_len(ncol(sv_feats))) {
  set(sv_feat, j = sprintf("tp_svd_%d", k - 1L), value = sv_feats[, k])
}
all_dt <- merge(all_dt, sv_feat, by = "rid___", all.x = TRUE, sort = FALSE)

# -----------------------------
# 4) Keyword category features (renamed)
# -----------------------------

In [ ]:
kw_dairy <- c(
  "dairy", "milk", "milking", "cow", "calf", "calf feeding", "calf rearing",
  "lactating", "lactation", "heifer", "unga dai", "yara maziwa", "transition cow",
  "dairy cow feeding", "dairy feed", "dairy housing", "dairy hygiene",
  "feeding a dairy", "feeding a lactating", "production of quality milk",
  "factors affecting milk", "dairy nutrition"
)
kw_poultry <- c(
  "poultry", "chicken", "kienyeji", "layer", "layers", "broiler",
  "feed for layers", "feeds for layers", "kienyeji chicken", "kienyeji poultry",
  "poultry feed", "poultry feeding", "poultry housing", "poultry management",
  "poultry products", "poultry mangt", "poultry mngt", "how to rear healthy chicken",
  "how to feed kienyeji", "how to feed layers", "biodeal poultry",
  "record keeping in poultry", "selling to choice meats"
)
kw_crop <- c(
  "maize", "bean", "beans", "crop", "weed", "pest", "harvest", "planting",
  "fertilizer", "asili fertilizer", "microp+", "seed variety", "seeds",
  "weed management", "pest and disease management in maize", "pest and disease management in crops",
  "improved maize", "post-harvest", "post harvest", "harvesting",
  "importance of choosing the right seed", "aflatoxin mitigation through good agricultural",
  "microp+ planting", "microp+ topdressing"
)
kw_health <- c(
  "health", "disease", "vaccination", "vaccinate", "vaccinating", "vaccinations",
  "deworming", "deworming your dairy", "veterinary", "treatment",
  "east coast fever", "ecf", "antimicrobial resistance",
  "importance of vaccination", "importance of vaccinating",
  "herd health", "dairy health", "poultry health", "poultry diseases",
  "control of external parasites", "prevent and treat cows against worms",
  "why you should vaccinate", "diseases in dairy"
)
kw_nutrition <- c(
  "feeding", "nutrition", "feed", "tyari", "mineral", "supplement",
  "animal nutrition", "pembe", "unga dairy feeds", "unga feed",
  "importance of mineral supplementation", "health and feeds", "hygiene and feeds",
  "poultry and dairy feeding with tyari", "poultry & dairy feeding with tyari",
  "dairy nutrition with tyari", "poultry feeding with tyari", "biodeal dairy",
  "fodder conservation", "silage", "mama silage bags", "silage making"
)
kw_breeding <- c(
  "breeding", "ai fails", "artificial insemination", "infertility",
  "natural mating", "disadvantages of natural mating", "disadvantages in natural mating",
  "how to succeed in breeding", "successfull breeding", "causes of infertility",
  "infertility in dairy cows", "some reasons why ai fails", "reasons why ai fails",
  "crv"
)
kw_ruminant <- c("sheep", "goat", "sheep and goat", "sheep & goat", "goat management", "goat rearing")
kw_digital <- c("digicow", "ndume app", "digital finance", "kcb", "app", "the benefits of ndume")
kw_business <- c("record keeping", "record keeping in dairy", "record keeping in poultry", "selling to choice meats")
kw_energy <- c("biogas", "sistema biogas", "benefits of sistema biogas", "benfits of sistema biogas", "clean energy")
kw_hygiene <- c("hygiene", "biosecurity", "milking hygiene", "dairy hygiene", "milking hygie", "personal protective equipment", "ppe", "biodeal products")
kw_livestock <- c(
  "livestock management", "animal management", "general livestock",
  "herd management", "livestock management practices",
  "cow management", "calf rearing for october", "kienyeji chicken rearing for october",
  "transition cow management"
)

cats <- list(
  dairy = kw_dairy, poultry = kw_poultry, crop = kw_crop, health = kw_health,
  nutrition = kw_nutrition, breeding = kw_breeding, ruminant = kw_ruminant,
  digital = kw_digital, business = kw_business, energy = kw_energy,
  hygiene = kw_hygiene, livestock = kw_livestock
)

hit_count <- function(tp_list, keys) {
  if (length(tp_list) == 0) return(0L)
  tl <- tolower(tp_list)
  sum(vapply(tl, function(one) any(vapply(keys, function(k) grepl(k, one, fixed = TRUE), logical(1))), logical(1)))
}

for (nm in names(cats)) {
  cnt_col <- paste0("ct_", nm)
  has_col <- paste0("hx_", nm)
  keys <- cats[[nm]]
  set(all_dt, j = cnt_col, value = vapply(all_dt$tp_vec, function(v) hit_count(v, keys), integer(1)))
  set(all_dt, j = has_col, value = as.integer(all_dt[[cnt_col]] > 0L))
}

cnt_cols <- paste0("ct_", names(cats))
has_cols <- paste0("hx_", names(cats))

cmat <- as.matrix(all_dt[, ..cnt_cols])
mx <- apply(cmat, 1, max)
idx <- max.col(cmat, ties.method = "first")
all_dt[, tp_cat_main := ifelse(mx == 0, 0L, as.integer(idx))]
all_dt[, tp_cat_n := rowSums(as.matrix(all_dt[, ..has_cols]))]

all_dt[, trn_cat_cnt := .N, by = .(trainer, tp_cat_main)]
all_dt[, ward_cat_cnt := .N, by = .(ward, tp_cat_main)]
all_dt[, cty_cat_cnt := .N, by = .(county, tp_cat_main)]

Warning message in `[.data.table`(all_dt, , ..has_cols):
“Both 'has_cols' and '..has_cols' exist in calling scope. Please remove the '..has_cols' variable in calling scope for clarity.”


# -----------------------------
# 5) Farmer history features
# -----------------------------

In [ ]:
setorder(all_dt, farmer_name, training_day)

all_dt[, visit_no := seq_len(.N), by = farmer_name]
all_dt[, prev_day := shift(training_day, 1L), by = farmer_name]
all_dt[, gap_days := as.integer(training_day - prev_day)]
all_dt[is.na(gap_days), gap_days := -99L]
all_dt[, first_evt := as.integer(visit_no == 1L)]

all_dt[, seen_trn := as.integer(seq_len(.N) > 1L), by = .(farmer_name, trainer)]
all_dt[, seen_grp := as.integer(seq_len(.N) > 1L), by = .(farmer_name, group_name)]

all_dt[, prev_in_7 := count_recent(training_day, 7L), by = farmer_name]
all_dt[, prev_in_90 := count_recent(training_day, 90L), by = farmer_name]
all_dt[, prev_in_120 := count_recent(training_day, 120L), by = farmer_name]

evt_topics <- all_dt[, .(
  farmer_name = farmer_name,
  ID = ID,
  training_day = training_day,
  topic = unique(unlist(tp_vec))
), by = .(ID, farmer_name, training_day)]
evt_topics <- evt_topics[!is.na(topic)]
evt_topics <- unique(evt_topics, by = c("ID", "farmer_name", "topic"))
setorder(evt_topics, farmer_name, training_day, ID, topic)

evt_topics[, prev_topic_n := seq_len(.N) - 1L, by = .(farmer_name, topic)]
evt_topics[, is_new_tp := as.integer(prev_topic_n == 0L)]

topic_roll <- evt_topics[
  ,
  .(
    new_tp_cnt = sum(is_new_tp),
    seen_tp_cnt = sum(prev_topic_n > 0L),
    evt_tp_uniq = uniqueN(topic)
  ),
  by = ID
]
all_dt <- merge(all_dt, topic_roll, by = "ID", all.x = TRUE, sort = FALSE)

roll_topic_history <- function(topic_sets) {
  seen <- character(0)
  past <- vector("list", length(topic_sets))
  cumu <- integer(length(topic_sets))
  for (i in seq_along(topic_sets)) {
    past[[i]] <- seen
    seen <- union(seen, topic_sets[[i]])
    cumu[i] <- length(seen)
  }
  list(cum_unique = cumu, past_set = past)
}

farmer_ev <- all_dt[, .(farmer_name, visit_no, tp_set = lapply(tp_vec, unique))]
setorder(farmer_ev, farmer_name, visit_no)

farmer_ev[, c("farm_uniq_cum", "past_set") := {
  tmp <- roll_topic_history(tp_set)
  list(tmp$cum_unique, tmp$past_set)
}, by = farmer_name]

all_dt <- merge(
  all_dt,
  farmer_ev[, .(farmer_name, visit_no, farm_uniq_cum, past_set)],
  by = c("farmer_name", "visit_no"),
  all.x = TRUE,
  sort = FALSE
)

all_dt[, new_tp_ratio := new_tp_cnt / pmax(1L, farm_uniq_cum)]
all_dt[, jacc_past := mapply(function(cur, past) {
  cur <- unique(cur)
  past <- unique(past)
  u <- union(cur, past)
  if (length(u) == 0) return(0)
  length(intersect(cur, past)) / length(u)
}, tp_vec, past_set)]
all_dt[, past_set := NULL]

all_dt[, ward_day_n := .N, by = .(ward, training_day)]
all_dt[, evt_size_bin := cut(
  ward_day_n,
  breaks = c(0, 5, 20, 100, 10000),
  labels = c(0, 1, 2, 3),
  right = TRUE
)]
all_dt[, evt_size_bin := as.integer(as.character(evt_size_bin))]

all_dt[, al_dairy := get("ct_dairy") * ave(get("ct_dairy"), trainer, FUN = mean)]
all_dt[, al_crop := get("ct_crop") * ave(get("ct_crop"), trainer, FUN = mean)]

all_dt[, prev_day := NULL]
all_dt[, trn_ward_n := .N, by = .(trainer, ward)]
all_dt[, ward_n_all := .N, by = ward]
all_dt[, trn_ward_share := trn_ward_n / ward_n_all]

all_dt[, tp_complex0 := 0]
all_dt[, enc_tp_text := lex_encode0(vapply(tp_vec, function(v) paste(v, collapse = ", "), character(1)))]

# -----------------------------
# 6) Time and aggregate stats
# -----------------------------

In [ ]:
all_dt[, mth := month(training_day)]
all_dt[, mth_sin := sin(2 * pi * mth / 12)]
all_dt[, mth_cos := cos(2 * pi * mth / 12)]
all_dt[, is_peak_m := as.integer(mth %in% c(3, 4, 10, 11))]
all_dt[, is_off_m := as.integer(mth %in% c(6, 7, 12, 1))]
all_dt[, qtr := ((mth - 1L) %/% 3L) + 1L]

all_dt[, tp_len := lengths(tp_vec)]
all_dt[, trn_tp_mean1 := ave(tp_len, trainer, FUN = mean)]

uniq_len <- vapply(all_dt$tp_vec, function(v) length(unique(v)), integer(1))
all_dt[, trn_tp_divm := ave(uniq_len, trainer, FUN = mean)]
all_dt[, cat_tp_mean := ave(tp_len, tp_cat_main, FUN = mean)]

entropy_one <- function(v) {
  if (length(v) == 0) return(0)
  tab <- table(v)
  p <- as.numeric(tab) / sum(tab)
  -sum(p * log2(p))
}
all_dt[, tp_entropy := vapply(tp_vec, entropy_one, numeric(1))]
all_dt[, trn_tp_rate := trn_tp_mean1 / (tp_len + 1e-6)]

# -----------------------------
# 7) Encode categorical + aggregates
# -----------------------------

In [ ]:
for (cc in cat_cols) set(all_dt, j = cc, value = lex_encode0(all_dt[[cc]]))

for (gcol in c("county", "subcounty", "ward")) {
  ncol <- paste0("geo_", gcol, "_n")
  rcol <- paste0("geo_", gcol, "_coopm")
  all_dt[, (ncol) := .N, by = get(gcol)]
  all_dt[, (rcol) := mean(belong_to_cooperative), by = get(gcol)]
}

trn_stat <- all_dt[
  ,
  .(
    trn_evt_n = .N,
    trn_tp_mean2 = mean(tp_len),
    trn_trained_rate = mean(has_topic_trained_on)
  ),
  by = trainer
]
all_dt <- merge(all_dt, trn_stat, by = "trainer", all.x = TRUE, sort = FALSE)

for (g in c("trainer", "county", "ward", "group_name")) {
  nm <- paste0("evt_n_", g)
  all_dt[, (nm) := .N, by = get(g)]
}

all_dt[, grp_n := .N, by = group_name]
all_dt[, grp_coop_m := mean(belong_to_cooperative), by = group_name]
all_dt[, grp_tp_mean := mean(tp_len), by = group_name]
all_dt[, grp_trained_m := mean(has_topic_trained_on), by = group_name]

all_dt[, enc_age_sex := lex_encode0(paste(age, gender, sep = "_"))]
all_dt[, enc_coop_trn := lex_encode0(paste(belong_to_cooperative, trainer, sep = "_"))]
all_dt[, enc_reg_coop := lex_encode0(paste(registration, belong_to_cooperative, sep = "_"))]

# -----------------------------
# 8) Finalize matrices
# -----------------------------

In [ ]:
all_dt[, topics_list := NULL]
all_dt[, tp_vec := NULL]
all_dt[, tp_text_blob := NULL]

train_dt <- all_dt[is_train == 1L]
test_dt <- all_dt[is_train == 0L]

train_dt[, is_train := NULL]
test_dt[, c("is_train", targets) := list(NULL, NULL, NULL, NULL)]

train_dt[, rid___ := NULL]
test_dt[, rid___ := NULL]

train_dt[, training_day := NULL]
test_dt[, training_day := NULL]

drop_cols <- c(targets, "ID", "farmer_name", "topic_list")
feat_cols <- setdiff(names(train_dt), drop_cols)

X_train_all <- as.matrix(train_dt[, ..feat_cols])
X_test_all <- as.matrix(test_dt[, ..feat_cols])
y_all <- train_dt[, ..targets]

# -----------------------------
# 9) Model training (same hyperparams)
# -----------------------------

In [ ]:
lgb_params <- list(
  num_iterations = 1500L,
  learning_rate = 0.08,
  max_depth = 4L,
  feature_fraction = 0.6,
  bagging_fraction = 0.7,
  min_data_in_leaf = 20L,
  lambda_l1 = 0.1,
  lambda_l2 = 0.1,
  verbosity = -1,
  objective = "binary",
  metric = "binary_logloss"
)

xgb_params <- list(
  nrounds = 900,
  eta = 0.01,
  max_depth = 8L,
  colsample_bytree = 0.7,
  subsample = 0.8,
  objective = "binary:logistic",
  eval_metric = "logloss"
)

grp <- train_dt$farmer_name
NF <- 5L
fold_groups <- make_group_folds(grp, k = NF, seed = 1L)

pred_auc <- list()
pred_log <- list()

log_loss_metric <- function(y, p) {
  p <- pmin(pmax(p, 1e-15), 1 - 1e-15)
  -mean(y * log(p) + (1 - y) * log(1 - p))
}

for (tgt in targets) {
  xgb_fold_preds <- vector("list", NF)
  lgb_fold_preds <- vector("list", NF)
  y_vec <- as.numeric(y_all[[tgt]])

  oof_xgb <- numeric(nrow(X_train_all))
  oof_lgb <- numeric(nrow(X_train_all))

  for (f in 1:NF) {
    val_g <- fold_groups[[f]]
    val_idx <- which(grp %in% val_g)
    tr_idx <- setdiff(seq_along(grp), val_idx)

    Xtr <- X_train_all[tr_idx, , drop = FALSE]
    Xva <- X_train_all[val_idx, , drop = FALSE]
    ytr <- y_vec[tr_idx]
    yva <- y_vec[val_idx]

    dtr <- xgb.DMatrix(Xtr, label = ytr)
    dva <- xgb.DMatrix(Xva, label = yva)
    dte <- xgb.DMatrix(X_test_all)

    prm_x <- modifyList(xgb_params, list(seed = as.integer(f + 1L)))
    xgb_mod <- xgb.train(
      params = prm_x,
      data = dtr,
      nrounds = prm_x$nrounds,
      watchlist = list(val = dva),
      verbose = 0
    )
    xgb_fold_preds[[f]] <- predict(xgb_mod, dte)
    oof_xgb[val_idx] <- predict(xgb_mod, dva)

    dtr_l <- lgb.Dataset(data = Xtr, label = ytr)
    dva_l <- lgb.Dataset(data = Xva, label = yva)

    prm_l <- modifyList(lgb_params, list(seed = as.integer(f + 1L)))
    lgb_mod <- lgb.train(
      params = prm_l,
      data = dtr_l,
      nrounds = prm_l$num_iterations,
      valids = list(val = dva_l),
      early_stopping_rounds = 50L,
      verbose = -1
    )
    lgb_fold_preds[[f]] <- predict(lgb_mod, X_test_all)
    oof_lgb[val_idx] <- predict(lgb_mod, Xva)
  }

  cat(sprintf("Target: %s\n", tgt))
  cat(sprintf("  OOF XGB LogLoss: %.6f\n", log_loss_metric(y_vec, oof_xgb)))
  cat(sprintf("  OOF LGB LogLoss: %.6f\n", log_loss_metric(y_vec, oof_lgb)))

  xgb_mean <- Reduce(`+`, xgb_fold_preds) / NF
  lgb_mean <- Reduce(`+`, lgb_fold_preds) / NF

  if (tgt == "adopted_within_07_days") {
    pred_auc[[tgt]] <- 0.5 * xgb_mean + 0.5 * lgb_mean
    pred_log[[tgt]] <- 0.3 * xgb_mean + 0.7 * lgb_mean
  } else if (tgt == "adopted_within_90_days") {
    pred_auc[[tgt]] <- 0.6 * xgb_mean + 0.4 * lgb_mean
    pred_log[[tgt]] <- 0.5 * xgb_mean + 0.5 * lgb_mean
  } else {
    pred_auc[[tgt]] <- 0.6 * xgb_mean + 0.4 * lgb_mean
    pred_log[[tgt]] <- 0.55 * xgb_mean + 0.5 * lgb_mean
  }

}

Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in

Target: adopted_within_07_days
  OOF XGB LogLoss: 0.035849
  OOF LGB LogLoss: 0.035864


Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in

Target: adopted_within_90_days
  OOF XGB LogLoss: 0.070060
  OOF LGB LogLoss: 0.070178


Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'watchlist' has been renamed to 'evals'. This warning will become an error in

Target: adopted_within_120_days
  OOF XGB LogLoss: 0.095318
  OOF LGB LogLoss: 0.094792


# -----------------------------
# 10) Build submission
# -----------------------------

In [ ]:

subm <- data.table(ID = test_dt$ID)

for (tgt in targets) {
  subm[[paste0(tgt, "_AUC")]] <- pred_auc[[tgt]]
  subm[[paste0(tgt, "_LogLoss")]] <- pred_log[[tgt]]
}

subm <- prob_sharpen(subm, gamma = 1.2, eps = 0.0000000000000001)

setnames(
  subm,
  old = c(
    "adopted_within_07_days_AUC", "adopted_within_90_days_AUC", "adopted_within_120_days_AUC",
    "adopted_within_07_days_LogLoss", "adopted_within_90_days_LogLoss", "adopted_within_120_days_LogLoss"
  ),
  new = c(
    "Target_07_AUC", "Target_90_AUC", "Target_120_AUC",
    "Target_07_LogLoss", "Target_90_LogLoss", "Target_120_LogLoss"
  )
)

subm[, Target_90_AUC := pmax(Target_07_AUC, Target_90_AUC)]
subm[, Target_120_AUC := pmax(Target_90_AUC, Target_120_AUC)]
subm[, Target_90_LogLoss := pmax(Target_07_LogLoss, Target_90_LogLoss)]
subm[, Target_120_LogLoss := pmax(Target_90_LogLoss, Target_120_LogLoss)]

fwrite(subm, "finsub.csv")